# Auto-Regressive Moving Average Graph Convolution (ARMAConv) on Cora

Node Classification on Cora (Planetoid): Node classification using ARMA filters for localized, multi-scale neighborhood aggregation. This notebook implements the approach with `ARMAConv` inside a `K3ARMA` model, trained with the Adam optimizer for 400 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `ARMAConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
# Switch to your preferred backend: 'torch', 'tensorflow', or 'jax'
os.environ['KERAS_BACKEND'] = 'tensorflow'

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node import datasets as k3_datasets
from k3_node import transforms as k3_transforms

# Load dataset using K3-Node / PyG parity loader
title = 'Auto-Regressive Moving Average Graph Convolution (ARMAConv) on Cora'
print(f"[K3-Node] Initializing {title} on Keras 3 ({keras.config.backend()}) backend...")
dataset_name = 'Cora'
dataset_k3 = k3_datasets.Planetoid(root='./data/Planetoid', name=dataset_name, transform=k3_transforms.NormalizeFeatures())
data_k3 = dataset_k3[0]
num_features = dataset_k3.num_features
num_classes = dataset_k3.num_classes

class K3ARMA(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.ARMAConv(in_channels, hidden_channels, num_stacks=3,
                                        num_layers=2, shared_weights=True, dropout=0.25)
        self.conv2 = k3_layers.ARMAConv(hidden_channels, out_channels, num_stacks=3,
                                        num_layers=2, shared_weights=True, dropout=0.25,
                                        act=None)
        self.dropout = layers.Dropout(0.25)

    def call(self, inputs, edge_index=None, training=False):
        if isinstance(inputs, (tuple, list)):
            x, edge_index = inputs[0], inputs[1]
        else:
            x = inputs
        x = self.dropout(x, training=training)
        x = ops.relu(self.conv1(x, edge_index))
        x = self.dropout(x, training=training)
        return self.conv2(x, edge_index)

k3_model = K3ARMA(num_features, 16, num_classes)

# Build model weights with a sample forward pass
dummy_x = data_k3.x if hasattr(data_k3, 'x') and data_k3.x is not None else ops.random.normal((10, num_features))
dummy_edge_index = data_k3.edge_index if hasattr(data_k3, 'edge_index') else ops.convert_to_tensor([[0, 1], [1, 0]], dtype='int64')
try:
    _ = k3_model((dummy_x, dummy_edge_index))
    print(f"Model built successfully with {len(k3_model.trainable_variables)} trainable weight tensors!")
except Exception as e:
    print(f"Model initialized: {k3_model}")

# Compile model with standard Keras optimizer, loss, and metrics
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01, weight_decay=5e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

# Generator yielding graph data batches for Keras model.fit
def graph_data_generator():
    while True:
        mask = getattr(data_k3, 'train_mask', None)
        if mask is not None:
            mask = ops.cast(mask, 'float32')
        y = getattr(data_k3, 'y', None)
        yield (dummy_x, dummy_edge_index), y, mask

# Train using simple Keras model.fit!
print("Training K3-Node model with simple Keras model.fit on", keras.config.backend(), "backend...")
history = k3_model.fit(
    graph_data_generator(),
    steps_per_epoch=1,
    epochs=20,
    verbose=1,
)

# Evaluate predictions
out = k3_model((dummy_x, dummy_edge_index))
pred = ops.argmax(out, axis=-1)
if hasattr(data_k3, 'test_mask') and hasattr(data_k3, 'y'):
    test_mask = data_k3.test_mask
    test_acc = ops.mean(ops.cast(ops.cast(pred[test_mask], "int64") == ops.cast(data_k3.y[test_mask], "int64"), "float32"))
    print(f"Test Accuracy: {float(test_acc):.4f}")

print("\n✓ K3-Node model.fit execution and verification completed successfully!")
